# Syteline (Infor CSI) -> Eventstream Ingestion

Metadata-driven ingestion from Infor CloudSuite Industrial (Syteline) into Microsoft
Fabric: pulls entity data via the **ION REST IDO API** and produces **CloudEvents** to a
**schema-associated Eventstream custom endpoint**. Job definitions and per-entity
incremental watermarks live in a Warehouse control table, so adding an entity is a
control-row registration - not a notebook edit. Events go out over either **AMQP**
(azure-eventhub SDK) or the **Event Hubs REST batch API** (plain `requests`, no SDK
install), selected by the `SEND_PROTOCOL` parameter.

This is a **pure Python notebook** (no Spark). Unlike the other notebooks in this folder
it does not land data itself - the Eventstream (and whatever it routes to, e.g. an
Eventhouse) owns the landing. The pattern generalizes to any REST source: swap the
*IDO load & transform* cell for your source's paging API and keep the control-plane and
producer cells unchanged.

**Usage:**
1. Create the control-plane objects in a Fabric Warehouse (see *Control-plane contract*)
   and register at least one entity.
2. Store the secrets in Azure Key Vault (see *Configuration* - four ION credentials plus
   one Eventstream SAS key per environment).
3. Fill the parameters cell and the `<Placeholders>` in Configuration - or leave the
   parameters blank and define matching variables in a workspace **Variable Library**;
   blanks resolve from its active value set at run time.
4. Run with `DRY_RUN = True` first: it loads, transforms, and prints one sample
   CloudEvents envelope without sending anything, advancing watermarks, or logging.

**Auth:** ION uses an OAuth2 resource-owner (password) grant - SAAK/SASK service-account
keys as username/password plus a backing-service client id/secret, all four values from
the tenant's `.ionapi` credential file. The Warehouse connection uses the notebook
identity's Entra token over TDS. The Eventstream custom endpoint uses a SAS key; only the
key itself lives in Key Vault - the rest of the connection string is assembled from
config.

**Scheduling:** invoke from a Data Pipeline Notebook activity that passes every parameter
explicitly (e.g. from a Variable Library `libraryVariables` block), including
`DRY_RUN = false` and the `ENTITY_FILTER` / `SEND_PROTOCOL` knobs, so scheduled runs go
live without editing this notebook. Parameter-complete runs never call
`notebookutils.variableLibrary` - which matters because that API has no
service-principal support.

## Control-plane contract

One Warehouse schema (`ingest`) carries the control table, a run log, and their procs.
The notebook reads `ingest.Control`, inserts `ingest.RunLog` telemetry via
`ingest.usp_LogRun`, and advances `LastWatermark` directly (sequential per-entity
updates - safe because this notebook is not parallel).

```sql
CREATE TABLE ingest.Control (
    SourceSystemName varchar(20)  NOT NULL, -- e.g. 'Syteline-ION'
    SourceObjectName varchar(50)  NOT NULL, -- IDO name; doubles as CloudEvent type + schema name
    SchemaVersion    varchar(10)  NOT NULL, -- schema-registry version segment, e.g. 'v1'
    SourceWatermark  varchar(50)  NOT NULL, -- IDO property filtered for the incremental window
    SourceOrderBy    varchar(200) NULL,
    SourceFilter     varchar(max) NULL,     -- extra IDO filter ANDed onto the watermark window
    FieldMap         varchar(max) NOT NULL, -- JSON: [{"OutputFieldName": "...", "Sources": ["P(Prop)", "literal"]}]
    LastWatermark    datetime2(3) NULL,     -- run state: advanced on success only
    IsActive         bit          NOT NULL
);
ALTER TABLE ingest.Control ADD CONSTRAINT PK_IngestControl
    PRIMARY KEY NONCLUSTERED (SourceSystemName, SourceObjectName);
```

`ingest.usp_LogRun` inserts one row per entity per run (`@PipelineRunId`,
`@SourceSystemName`, `@SourceObjectName`, `@Status`, `@RecordsFetched`, `@RecordsSent`,
`@DurationSeconds`, `@Watermark`, `@FailureMessage`). Register control rows through an
idempotent stored procedure rather than raw INSERTs, and never overwrite `LastWatermark`
on re-registration - it is run state, not config; rewinding it re-copies the whole window.

`FieldMap` sources: `P(PropertyName)` resolves to the IDO property; any other string is a
literal. Sources concatenate in order into the output field - all values emit as strings.

## Parameters (pipeline-overridable)

Mark this as the notebook's **parameter cell** after importing into Fabric (cell toolbar
-> *Toggle parameter cell*; the tag does not survive metadata scrubbing). A pipeline
Notebook activity overrides these at submission. Blank values resolve from the workspace
Variable Library named in Configuration (active value set), so an interactive run in any
workspace picks up that workspace's config with no edits here. `DRY_RUN` ships closed
for interactive safety; the pipeline passes `false` plus the run-scoping knobs for live
scheduled runs.

In [ ]:
# DRY_RUN  True (default) - load + transform + report per-entity counts and one sample
#          envelope; send nothing, advance no watermarks, log nothing.
#          The pipeline passes false for live scheduled runs.
DRY_RUN = True
# Comma-separated SourceObjectName list restricting the run (e.g. "SLTerms,SLCarriers");
# blank = every active control row. A string, not a list - notebook parameters are
# scalars only - parsed to the list the run loop consumes in Configuration.
ENTITY_FILTER = ""
# CloudEvents transport: "https" (Event Hubs REST batch send; no SDK install) or "amqp"
# (azure-eventhub SDK). Validated in Configuration.
SEND_PROTOCOL = "https"
# Everything below: blank = resolved from the Variable Library in Configuration (variable
# named in the comment). Pipelines should pass all of these explicitly - parameter-complete
# runs never hit the variableLibrary API (same-workspace only, no SP support).
WAREHOUSE_SQL_SERVER = ""  # WarehouseSqlEndpoint - <guid>.datawarehouse.fabric.microsoft.com
WAREHOUSE_DB = ""          # WarehouseName - item display name (TDS Initial Catalog rule)
PIPELINE_RUN_ID = ""       # @pipeline().RunId when pipeline-invoked; blank mints a GUID
KEY_VAULT_URI = ""         # AzureKeyVaultUri
ENVIRONMENT_SUFFIX = ""    # EnvironmentSuffix - lowercased into env-specific secret names
EVENTHUB_NAMESPACE = ""    # EventhubNamespace - the endpoint's eseh* "Event hub name"
EVENTHUB_KEY_NAME = ""     # EventstreamSharedAccessKeyName - the key_<guid> SAS policy name
SCHEMA_REGISTRY_HOST = ""    # EventSchemaRegistryHost - rthprod* messagingcatalog host label
SCHEMA_REGISTRY_REGION = ""  # EventSchemaRegistryRegion - e.g. eastus2
SCHEMA_GROUP_ID = ""         # ItemReferenceSchemaSet.itemId (schema set item id = group id)

## Configuration

Blank parameters resolve here from the workspace Variable Library's active value set.
Secrets live in Key Vault and are fetched with `notebookutils.credentials.getSecret` -
nothing secret is stored in the notebook or the control table. Secret-naming contract:
environment-specific secrets append the lowercased `ENVIRONMENT_SUFFIX`
(e.g. `eventstream-shared-access-key-prod`); shared credentials are unsuffixed.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────────
# --- Environment resolution (hard fail) ---
# Any parameter left blank resolves from this workspace's Variable Library (active value
# set). Lazy: a fully parameterized (pipeline) run makes no variableLibrary calls at all.
import notebookutils  # imported early (before the Imports cell) for resolution

VARIABLE_LIBRARY_NAME = "<VariableLibraryName>"


def vl_lookup(variable_name: str):
    """Resolve one Variable Library variable from the workspace's active value set.

    Args:
        variable_name: Variable name in the library (case-sensitive).

    Returns:
        The typed variable value.

    Raises:
        RuntimeError: If the library or variable can't be resolved here.
    """
    try:
        return notebookutils.variableLibrary.get(
            f"$(/**/{VARIABLE_LIBRARY_NAME}/{variable_name})"
        )
    except Exception as exc:
        raise RuntimeError(
            f"Variable Library lookup failed for '{variable_name}'. Check that "
            f"'{VARIABLE_LIBRARY_NAME}' exists in this workspace and this runtime supports "
            "notebookutils.variableLibrary - or fill the value in the parameters cell."
        ) from exc


WAREHOUSE_SQL_SERVER = WAREHOUSE_SQL_SERVER or vl_lookup("WarehouseSqlEndpoint")
WAREHOUSE_DB = WAREHOUSE_DB or vl_lookup("WarehouseName")
KEY_VAULT_URI = KEY_VAULT_URI or vl_lookup("AzureKeyVaultUri")
ENVIRONMENT_SUFFIX = ENVIRONMENT_SUFFIX or vl_lookup("EnvironmentSuffix")
EVENTHUB_NAMESPACE = EVENTHUB_NAMESPACE or vl_lookup("EventhubNamespace")
EVENTHUB_KEY_NAME = EVENTHUB_KEY_NAME or vl_lookup("EventstreamSharedAccessKeyName")
SCHEMA_REGISTRY_HOST = SCHEMA_REGISTRY_HOST or vl_lookup("EventSchemaRegistryHost")
SCHEMA_REGISTRY_REGION = SCHEMA_REGISTRY_REGION or vl_lookup("EventSchemaRegistryRegion")
if not SCHEMA_GROUP_ID:
    # ItemReference resolution differs by kernel: the pure Python kernel returns a plain
    # dict-like object (.get("itemId") -> str), while the documented Spark surface wraps
    # members in .value() accessors. Accept both.
    _schema_set_item_id = vl_lookup("ItemReferenceSchemaSet").get("itemId")
    if callable(getattr(_schema_set_item_id, "value", None)):
        _schema_set_item_id = _schema_set_item_id.value()
    SCHEMA_GROUP_ID = _schema_set_item_id

# --- Run scoping ---
# STRICT: True raises after the summary if any entity failed (CI / orchestration).
STRICT = False
# ENTITY_FILTER arrives as a comma-separated string from the parameters cell; parse to
# the list the run loop consumes. Empty = every active control row. A blank pipeline
# parameter is injected as None (Fabric nulls empty string parameters), so guard it.
ENTITY_FILTER: list = [e.strip() for e in (ENTITY_FILTER or "").split(",") if e.strip()]

# --- Control plane ---
SOURCE_SYSTEM = "Syteline-ION"
INGEST_TABLE = "ingest.Control"
DEFAULT_WATERMARK = "1900-01-01T00:00:00"  # backfill start when LastWatermark is NULL

# --- Key Vault secret names (names only - values stay in the vault) ---
SECRET_ION_CLIENT_ID = "ion-client-id"          # .ionapi "ci"
SECRET_ION_CLIENT_SECRET = "ion-client-secret"  # .ionapi "cs"
SECRET_ION_SAAK = "ion-saak"                    # .ionapi "saak" (service account access key)
SECRET_ION_SASK = "ion-sask"                    # .ionapi "sask" (service account secret key)
SECRET_EVENTSTREAM_KEY = f"eventstream-shared-access-key-{ENVIRONMENT_SUFFIX.lower()}"

# --- ION API (non-secret; values come from the tenant's .ionapi credential file) ---
ION_TENANT = "<IonTenantId>"                                # .ionapi "ti"
ION_API_BASE = "https://mingle-ionapi.inforcloudsuite.com"  # .ionapi "iu"
ION_TOKEN_URL = f"https://mingle-sso.inforcloudsuite.com/{ION_TENANT}/as/token.oauth2"
ION_IDO_SUITE = "CSI"  # IDO endpoints live under {iu}/{ti}/CSI
# Syteline (Mongoose) configuration name - sent as the X-Infor-MongooseConfig header on
# every IDO call.
ION_MONGOOSE_CONFIG = "<MongooseConfigName>"
# Syteline stores RecordDate in server-local time; shift the incremental window to match.
SYTELINE_UTC_OFFSET_HOURS = 0

# --- Eventstream custom endpoint / schema registry ---
# The custom endpoint's hub is always <namespace>_eh (per its sample connection string),
# so EntityPath is derived, not configured.
EVENTHUB_ENTITY_PATH = f"{EVENTHUB_NAMESPACE}_eh"

# Transport for the CloudEvents producer (SEND_PROTOCOL comes from the parameters cell;
# see the producer cell for the wire details):
#   "https" - Event Hubs REST batch send over :443; no SDK install needed.
#   "amqp"  - azure-eventhub SDK over AMQP/TLS :5671.
# Same None guard as ENTITY_FILTER: a blank pipeline parameter falls back to the default.
SEND_PROTOCOL = SEND_PROTOCOL or "https"
if SEND_PROTOCOL not in ("amqp", "https"):
    raise RuntimeError(f"SEND_PROTOCOL must be 'amqp' or 'https', got '{SEND_PROTOCOL}'")

# REST batch requests must stay under the Event Hubs 1 MB message-size cap; headroom for
# the JSON envelope around each event.
HTTPS_BATCH_LIMIT_BYTES = 900 * 1024

# Schema-group URI = registry host + region + the schema set's item id (which doubles as
# the schema-group id). Per event, build_event_properties appends
# /schemas/<SourceObjectName>/versions/<SchemaVersion> from the control row.
SCHEMA_GROUP_URI = (
    f"https://{SCHEMA_REGISTRY_HOST}.{SCHEMA_REGISTRY_REGION}.messagingcatalog.azure.net"
    f"/schemagroups/{SCHEMA_GROUP_ID}"
)

# Every CloudEvent's source attribute. Any non-empty URI works; identify this producer.
EVENT_SOURCE = "urn:example:fabric:nb_syteline_ingest"

# Records per IDO page; the load loop follows MoreRowsExist/Bookmark until the window drains.
IDO_PAGE_SIZE = 5000


# Fail fast on unfilled configuration - a gap would otherwise surface as an opaque
# auth/DNS error mid-run. Blank, "FILL-ME" (Variable Library can't hold empty strings, so
# use a sentinel there), and <Placeholder> values all count as unfilled.
def _unfilled(value) -> bool:
    text = str(value)
    return not value or text == "FILL-ME" or (text.startswith("<") and text.endswith(">"))


_required = {
    "WAREHOUSE_SQL_SERVER": WAREHOUSE_SQL_SERVER,
    "WAREHOUSE_DB": WAREHOUSE_DB,
    "KEY_VAULT_URI": KEY_VAULT_URI,
    "ENVIRONMENT_SUFFIX": ENVIRONMENT_SUFFIX,
    "EVENTHUB_NAMESPACE": EVENTHUB_NAMESPACE,
    "EVENTHUB_KEY_NAME": EVENTHUB_KEY_NAME,
    "SCHEMA_REGISTRY_HOST": SCHEMA_REGISTRY_HOST,
    "SCHEMA_REGISTRY_REGION": SCHEMA_REGISTRY_REGION,
    "SCHEMA_GROUP_ID": SCHEMA_GROUP_ID,
    "ION_TENANT": ION_TENANT,
    "ION_MONGOOSE_CONFIG": ION_MONGOOSE_CONFIG,
}
_missing = [name for name, value in _required.items() if _unfilled(value)]
if _missing:
    raise RuntimeError(
        f"Unfilled configuration: {', '.join(_missing)}. Fill the parameters cell and the "
        "<Placeholders> above, or define the matching Variable Library variables."
    )

# Canonical result buckets - populated by the run loop below.
results = {
    "succeeded": [],  # list[str] of entity names
    "skipped":   [],  # list[dict]: {"name": str, "reason": str}
    "failed":    [],  # list[dict]: {"name": str, "error": str}
}

## Imports & dependency check

`azure-eventhub` is not in the default runtime, and pure Python notebooks have no
Environment-item library management - so the import self-installs it into the session on
first failure (adds ~15-30 s to a cold session; no-op once cached). Only the AMQP
transport needs it: `SEND_PROTOCOL = "https"` sends with `requests` and skips the
install entirely.

In [ ]:
import base64
import hashlib
import hmac
import json
import struct
import time
import urllib.parse
import uuid
from datetime import datetime, timedelta, timezone
from typing import Any, Dict, List, Optional

import pandas as pd
import pyodbc
import requests

if SEND_PROTOCOL == "amqp":
    try:
        from azure.eventhub import EventData, EventHubProducerClient
    except ImportError:
        # Pure Python notebooks have no Environment-item library management; self-install
        # into the session so interactive AND scheduled runs work without a manual %pip step.
        import subprocess
        import sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "azure-eventhub"])
        from azure.eventhub import EventData, EventHubProducerClient

## Authentication (hard fail)

Fetches the four ION credentials + the environment's Eventstream SAS key from Key Vault,
then acquires an ION bearer token. The Event Hubs connection string is assembled from the
config parts (namespace / key name / entity path) plus that key - only the key is secret
material. ION uses an OAuth2 resource-owner grant: SAAK/SASK as username/password plus
the backing-service client id/secret.

In [ ]:
def get_secret(name: str) -> str:
    """Fetch a secret from Key Vault, raising if it is missing or empty.

    Args:
        name: Secret name in the vault at KEY_VAULT_URI.

    Returns:
        The secret value.

    Raises:
        RuntimeError: If the secret cannot be fetched or is empty.
    """
    value = notebookutils.credentials.getSecret(KEY_VAULT_URI, name)
    if not value:
        raise RuntimeError(f"Key Vault secret '{name}' is empty or missing at {KEY_VAULT_URI}")
    return value


def acquire_ion_token() -> str:
    """Acquire an ION API bearer token via the OAuth2 password (SAAK/SASK) grant.

    Returns:
        The access token string.

    Raises:
        RuntimeError: On a non-200 token response, with status and body excerpt.
    """
    response = requests.post(
        ION_TOKEN_URL,
        data={
            "grant_type": "password",
            "username": get_secret(SECRET_ION_SAAK),
            "password": get_secret(SECRET_ION_SASK),
            "client_id": get_secret(SECRET_ION_CLIENT_ID),
            "client_secret": get_secret(SECRET_ION_CLIENT_SECRET),
        },
        timeout=60,
    )
    if response.status_code != 200:
        raise RuntimeError(
            f"ION token request failed: HTTP {response.status_code} - {response.text[:300]}"
        )
    token = response.json().get("access_token")
    if not token:
        raise RuntimeError("ION token response contained no access_token")
    return token


def ion_get(url: str, params: Dict[str, Any]) -> requests.Response:
    """GET against the ION API, re-acquiring the token once on a 401.

    A full backfill can outlive a single ION token (~2 h lifetime), so a mid-run 401
    triggers one transparent refresh + retry.

    Args:
        url: Full request URL.
        params: Query parameters.

    Returns:
        The (possibly retried) response; status handling stays with the caller.
    """
    response = ion_session.get(url, params=params, timeout=300)
    if response.status_code == 401:
        ion_session.headers["Authorization"] = f"Bearer {acquire_ion_token()}"
        response = ion_session.get(url, params=params, timeout=300)
    return response


ion_session = requests.Session()
ion_session.headers.update({
    "Authorization": f"Bearer {acquire_ion_token()}",
    "Accept": "application/json",
    "X-Infor-MongooseConfig": ION_MONGOOSE_CONFIG,  # required by Mongoose on every IDO call
})
# The raw key signs SAS tokens on the HTTPS path; the AMQP path consumes it inside the
# connection string. Either way it is the only secret part - everything else is convention.
eventstream_key = get_secret(SECRET_EVENTSTREAM_KEY)
eventstream_conn = (
    f"Endpoint=sb://{EVENTHUB_NAMESPACE}.servicebus.windows.net/;"
    f"SharedAccessKeyName={EVENTHUB_KEY_NAME};"
    f"SharedAccessKey={eventstream_key};"
    f"EntityPath={EVENTHUB_ENTITY_PATH}"
)
print("ION token acquired; Eventstream connection string assembled "
      f"({EVENTHUB_NAMESPACE} / {ENVIRONMENT_SUFFIX}).")

## Control plane: job definitions + watermarks (hard fail)

`ingest.Control` is the source of truth: one row per entity carrying the IDO name (which
doubles as the CloudEvent type), schema version, watermark property, and the `FieldMap`
JSON. `LastWatermark` is runtime state this notebook advances on success. Warehouse
access is via **pyodbc + an Entra token** (`notebookutils.credentials.getToken` with the
SQL/TDS audience `https://database.windows.net/`, injected as `SQL_COPT_SS_ACCESS_TOKEN`).

In [ ]:
def parse_field_map(raw: str, entity: str) -> List[Dict[str, Any]]:
    """Parse and validate one entity's FieldMap JSON.

    Args:
        raw: JSON array of {OutputFieldName, Sources[]} objects.
        entity: Entity name, for error messages.

    Returns:
        The parsed field list.

    Raises:
        RuntimeError: If the JSON is invalid or the shape is wrong.
    """
    try:
        fields = json.loads(raw)
    except json.JSONDecodeError as exc:
        raise RuntimeError(f"Invalid FieldMap JSON for '{entity}': {exc}") from exc
    if not isinstance(fields, list) or not fields:
        raise RuntimeError(f"FieldMap for '{entity}' must be a non-empty JSON array")
    for field in fields:
        if "OutputFieldName" not in field or "Sources" not in field:
            raise RuntimeError(
                f"FieldMap entry for '{entity}' is missing OutputFieldName/Sources: {field}"
            )
    return fields


def wh_connect() -> pyodbc.Connection:
    """Open a pyodbc connection to the control-plane Warehouse with an Entra token.

    Token audience is the SQL/TDS resource (https://database.windows.net/); the token is
    injected pre-auth via SQL_COPT_SS_ACCESS_TOKEN (attr 1256) as a UTF-16-LE
    length-prefixed struct. A fresh connection per operation keeps long backfills immune
    to token expiry.

    Returns:
        An open connection; caller commits writes.

    Raises:
        RuntimeError: If no SQL Server ODBC driver is installed in the runtime.
    """
    driver = next(
        (d for d in sorted(pyodbc.drivers(), reverse=True)
         if "ODBC Driver" in d and "SQL Server" in d),
        None,
    )
    if driver is None:
        raise RuntimeError(f"No SQL Server ODBC driver found; installed: {pyodbc.drivers()}")
    token = notebookutils.credentials.getToken("https://database.windows.net/")
    token_bytes = token.encode("utf-16-le")
    token_struct = struct.pack(f"<I{len(token_bytes)}s", len(token_bytes), token_bytes)
    return pyodbc.connect(
        f"Driver={{{driver}}};Server=tcp:{WAREHOUSE_SQL_SERVER},1433;"
        f"Database={WAREHOUSE_DB};Encrypt=Yes;TrustServerCertificate=No",
        attrs_before={1256: token_struct},  # SQL_COPT_SS_ACCESS_TOKEN
    )


def wh_query(sql: str) -> pd.DataFrame:
    """Run a SELECT against the Warehouse and return the result as a DataFrame.

    Args:
        sql: The T-SQL statement.

    Returns:
        DataFrame with the statement's result set (empty if no rows).
    """
    with wh_connect() as conn:
        cursor = conn.cursor()
        cursor.execute(sql)
        columns = [d[0] for d in cursor.description]
        return pd.DataFrame.from_records(
            [tuple(row) for row in cursor.fetchall()], columns=columns
        )


def wh_execute(sql: str) -> int:
    """Run a write statement against the Warehouse and commit it.

    Args:
        sql: The T-SQL statement.

    Returns:
        Number of rows affected.
    """
    with wh_connect() as conn:
        cursor = conn.cursor()
        cursor.execute(sql)
        affected = cursor.rowcount
        conn.commit()
        return affected


def load_ingest_control() -> List[Dict[str, Any]]:
    """Read active ingest jobs (definitions + watermarks) from the control table.

    Returns:
        One dict per entity: entity, schema_version, watermark_property, order_by,
        filter, fields, last_watermark (datetime or None).

    Raises:
        RuntimeError: If no active rows exist or a FieldMap fails validation.
    """
    frame = wh_query(
        f"""
        SELECT
              SourceObjectName
            , SchemaVersion
            , SourceWatermark
            , SourceOrderBy
            , SourceFilter
            , FieldMap
            , LastWatermark
        FROM {INGEST_TABLE}
        WHERE IsActive = 1
            AND SourceSystemName = '{SOURCE_SYSTEM}'
        ORDER BY SourceObjectName;
        """
    )
    if frame is None or len(frame) == 0:
        raise RuntimeError(
            f"No active rows in {INGEST_TABLE} for SourceSystemName = '{SOURCE_SYSTEM}'. "
            "Register at least one entity first (see the control-plane contract above)."
        )
    jobs: List[Dict[str, Any]] = []

    def null_safe(value: Any) -> Any:
        """Collapse pandas NULL representations (None, NaT, NaN) to None."""
        return None if pd.isna(value) else value

    for row in frame.itertuples(index=False):
        last_watermark = null_safe(row.LastWatermark)
        if last_watermark is not None and hasattr(last_watermark, "to_pydatetime"):
            last_watermark = last_watermark.to_pydatetime()
        jobs.append({
            "entity": row.SourceObjectName,
            "schema_version": row.SchemaVersion,
            "watermark_property": row.SourceWatermark,
            "order_by": null_safe(row.SourceOrderBy),
            "filter": null_safe(row.SourceFilter),
            "fields": parse_field_map(row.FieldMap, row.SourceObjectName),
            "last_watermark": last_watermark,
        })
    return jobs


def set_watermark(entity: str, watermark: datetime) -> None:
    """Advance one entity's watermark (called only on success, never in DRY_RUN).

    Sequential per-entity UPDATEs are safe here because this notebook processes entities
    one at a time; a parallel orchestrator would need a set-based advance instead to
    avoid Warehouse write-write conflicts.

    Args:
        entity: SourceObjectName to update.
        watermark: New watermark value (UTC-naive).

    Raises:
        RuntimeError: If the UPDATE did not affect exactly one row.
    """
    safe_entity = entity.replace("'", "''")
    affected = wh_execute(
        f"""
        UPDATE {INGEST_TABLE}
        SET LastWatermark = '{watermark.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3]}'
        WHERE SourceSystemName = '{SOURCE_SYSTEM}'
            AND SourceObjectName = '{safe_entity}';
        """
    )
    if affected != 1:
        raise RuntimeError(f"Watermark update for '{entity}' affected {affected} rows")


RUN_ID = PIPELINE_RUN_ID or str(uuid.uuid4())


def log_run(
    entity: str,
    status: str,
    records_fetched: Optional[int] = None,
    records_sent: Optional[int] = None,
    duration_seconds: Optional[int] = None,
    watermark: Optional[datetime] = None,
    failure_message: Optional[str] = None,
) -> None:
    """Insert one ingest.RunLog row via ingest.usp_LogRun.

    Telemetry only - a logging failure prints a warning but never fails the entity
    (the run outcome is already captured in `results` and the watermark state).

    Args:
        entity: SourceObjectName the row describes.
        status: SUCCESS, NOROWS, or FAILED.
        records_fetched: Records returned by the IDO load.
        records_sent: Records delivered to the custom endpoint as CloudEvents.
        duration_seconds: Wall-clock seconds for the entity.
        watermark: Value the entity's watermark advanced to (success only).
        failure_message: Exception text for FAILED rows.
    """
    def sql_literal(value: Any) -> str:
        if value is None:
            return "NULL"
        if isinstance(value, int):
            return str(value)
        if isinstance(value, datetime):
            return f"'{value.strftime('%Y-%m-%dT%H:%M:%S.%f')[:-3]}'"
        return "'" + str(value).replace("'", "''") + "'"

    try:
        wh_execute(
            f"""
            EXEC ingest.usp_LogRun
                  @PipelineRunId = '{RUN_ID}'
                , @SourceSystemName = '{SOURCE_SYSTEM}'
                , @SourceObjectName = {sql_literal(entity)}
                , @Status = {sql_literal(status)}
                , @RecordsFetched = {sql_literal(records_fetched)}
                , @RecordsSent = {sql_literal(records_sent)}
                , @DurationSeconds = {sql_literal(duration_seconds)}
                , @Watermark = {sql_literal(watermark)}
                , @FailureMessage = {sql_literal(failure_message[:4000] if failure_message else None)};
            """
        )
    except Exception as exc:
        print(f"  WARNING: RunLog insert failed for {entity}: {exc}")


job_definitions = load_ingest_control()
print(f"Loaded {len(job_definitions)} active ingest jobs from {INGEST_TABLE} "
      f"(run {RUN_ID}): {', '.join(j['entity'] for j in job_definitions)}")

## IDO load & transform

GET `{iu}/{tenant}/CSI/IDORequestService/ido/load/{ido}` with `loadType=NEXT`, paging via
the response's `MoreRowsExist` / `Bookmark`, and the `X-Infor-MongooseConfig` header on
every call. This is the source-specific cell: to reuse the pattern for a different REST
source, replace it and keep everything else.

In [ ]:
def format_syteline_timestamp(value: datetime) -> str:
    """Format a watermark for an IDO filter clause.

    Syteline runs on SQL Server ``datetime`` (millisecond precision) in server-local
    time; SYTELINE_UTC_OFFSET_HOURS shifts the incremental window to compensate.

    Args:
        value: UTC-naive watermark timestamp.

    Returns:
        Timestamp string for embedding in the IDO filter.
    """
    shifted = value + timedelta(hours=SYTELINE_UTC_OFFSET_HOURS)
    return shifted.strftime("%Y-%m-%d %H:%M:%S")


def load_ido_records(job: Dict[str, Any], since: datetime) -> List[Dict[str, Any]]:
    """Stream all records for one entity from the IDO REST API since a watermark.

    Args:
        job: Ingest job dict from the control table.
        since: Lower bound for the incremental window (inclusive).

    Returns:
        Raw records as property-name -> value dicts.

    Raises:
        RuntimeError: On a non-200 IDO response.
    """
    entity = job["entity"]
    properties = sorted({
        src[2:-1]
        for field in job["fields"]
        for src in field["Sources"]
        if src.startswith("P(") and src.endswith(")")
    })
    filter_clause = f"{job['watermark_property']} >= '{format_syteline_timestamp(since)}'"
    if job["filter"]:
        filter_clause = f"({filter_clause}) AND ({job['filter']})"

    # The REST response carries Items / Bookmark / MoreRowsExist (PascalCase); the
    # Bookmark feeds the next page until MoreRowsExist goes false.
    url = f"{ION_API_BASE}/{ION_TENANT}/{ION_IDO_SUITE}/IDORequestService/ido/load/{entity}"
    records: List[Dict[str, Any]] = []
    bookmark: Optional[str] = None
    while True:
        params = {
            "properties": ",".join(properties),
            "filter": filter_clause,
            "recordCap": IDO_PAGE_SIZE,
            "loadType": "NEXT",
        }
        if job["order_by"]:
            params["orderBy"] = job["order_by"]
        if bookmark:
            params["bookmark"] = bookmark
        response = ion_get(url, params)
        if response.status_code != 200:
            raise RuntimeError(
                f"IDO load failed for {entity}: HTTP {response.status_code} - "
                f"{response.text[:300]}"
            )
        payload = response.json()
        records.extend(payload.get("Items", []))
        if not payload.get("MoreRowsExist"):
            break
        bookmark = payload.get("Bookmark")
        if not bookmark:
            break
        # Only multi-page pulls (backfills) narrate; steady-state increments stay quiet.
        print(f"    {entity}: {len(records):,} record(s) fetched so far...", flush=True)
    return records


# IDO date shapes normalized to ISO inside the transform - REST and SOAP surfaces return
# different date formats, and both must land as "s"-format ISO so downstream typed
# parsing (e.g. KQL todatetime()) never receives unparseable values.
_IDO_DATE_FORMATS = ("%Y%m%d %H:%M:%S.%f", "%m/%d/%Y %I:%M:%S %p")


def normalize_ido_value(value: Any) -> str:
    """Convert one raw IDO property value to its output string.

    Datetime-shaped strings normalize to ISO ("2026-06-03T13:02:03"); everything else
    passes through as str. None becomes "".

    Args:
        value: Raw property value from the IDO response.

    Returns:
        The normalized string value.
    """
    if value is None:
        return ""
    text = str(value)
    for fmt in _IDO_DATE_FORMATS:
        try:
            return datetime.strptime(text, fmt).strftime("%Y-%m-%dT%H:%M:%S")
        except ValueError:
            continue
    return text


def transform_record(record: Dict[str, Any], job: Dict[str, Any]) -> Dict[str, str]:
    """Map a raw IDO record to the job's output field shape.

    Each output field concatenates its sources: ``P(Name)`` resolves to the IDO property,
    anything else is a literal. All values are emitted as strings - pair with all-string
    landing tables and type downstream.

    Args:
        record: Raw IDO record (property name -> value).
        job: Ingest job dict.

    Returns:
        Output-field-name -> string-value dict, the CloudEvent data payload.
    """
    output: Dict[str, str] = {}
    for field in job["fields"]:
        pieces = []
        for src in field["Sources"]:
            if src.startswith("P(") and src.endswith(")"):
                pieces.append(normalize_ido_value(record.get(src[2:-1])))
            else:
                pieces.append(src)
        output[field["OutputFieldName"]] = "".join(pieces)
    return output

## CloudEvents producer

The wire contract for schema-associated custom endpoints (verified end-to-end):
**binary content mode** - the body is the payload JSON only, and every CloudEvents
attribute travels as a `cloudEvents:`-prefixed AMQP application property.
`cloudEvents:type` selects the schema (case-sensitive); `cloudEvents:dataschema` must pin
the **current** schema version or events are silently dropped by the schema-registry gate.

Two transports carry that same contract, selected by `SEND_PROTOCOL`:

- **amqp** - azure-eventhub SDK over AMQP/TLS :5671.
- **https** - Event Hubs REST *batch* send over :443, where the attributes ride as
  per-message `UserProperties` in the JSON request body (they map to AMQP application
  properties service-side). Batch mode is the only viable HTTP shape: the single-event
  API takes custom properties as HTTP headers, and `cloudEvents:*` names are illegal
  header tokens. Known contract gap: REST cannot set the per-message AMQP content-type
  (`cloudEvents:datacontenttype` still travels) - confirm via downstream read-back
  before trusting this path against a strict gate.

In [ ]:
def build_event_properties(job: Dict[str, Any]) -> Dict[str, str]:
    """Build the binary-mode CloudEvents attribute set for one event.

    A fresh id/time is minted per call - CloudEvents requires source + id to be unique.
    These attributes must travel beside the body (AMQP application properties or REST
    UserProperties), never inside it.

    Args:
        job: Ingest job dict (supplies event type + schema version).

    Returns:
        Property-name -> string-value dict, `cloudEvents:`-prefixed.
    """
    event_type = job["entity"]
    return {
        "cloudEvents:specversion": "1.0",
        "cloudEvents:type": event_type,
        "cloudEvents:source": EVENT_SOURCE,
        "cloudEvents:id": str(uuid.uuid4()),
        "cloudEvents:time": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "cloudEvents:datacontenttype": "application/json",
        "cloudEvents:dataschema": (
            f"{SCHEMA_GROUP_URI.rstrip('/')}/schemas/{event_type}/versions/"
            f"{job['schema_version']}"
        ),
    }


def build_event(job: Dict[str, Any], payload: Dict[str, str]) -> "EventData":
    """Wrap one output record as a binary-mode CloudEvent EventData (AMQP path only).

    Args:
        job: Ingest job dict (supplies event type + schema version).
        payload: Output-field dict produced by transform_record.

    Returns:
        EventData ready to add to a producer batch.
    """
    properties = build_event_properties(job)
    event = EventData(json.dumps(payload, ensure_ascii=False))
    event.content_type = "application/json"
    event.message_id = properties["cloudEvents:id"]
    event.properties = properties
    return event


def build_sas_token(resource_uri: str, ttl_seconds: int = 3600) -> str:
    """Sign a Service Bus SAS token for the HTTPS send path.

    Args:
        resource_uri: The entity URL the token authorizes (https://<ns>...net/<hub>).
        ttl_seconds: Token lifetime; one token covers a whole entity's send loop.

    Returns:
        The Authorization-header value (SharedAccessSignature sr=...&sig=...&se=...&skn=...).
    """
    encoded_uri = urllib.parse.quote_plus(resource_uri)
    expiry = str(int(time.time()) + ttl_seconds)
    signature = base64.b64encode(
        hmac.new(
            eventstream_key.encode("utf-8"),
            f"{encoded_uri}\n{expiry}".encode("utf-8"),
            hashlib.sha256,
        ).digest()
    )
    return (
        f"SharedAccessSignature sr={encoded_uri}"
        f"&sig={urllib.parse.quote_plus(signature)}&se={expiry}&skn={EVENTHUB_KEY_NAME}"
    )


def send_records_amqp(job: Dict[str, Any], payloads: List[Dict[str, str]]) -> int:
    """Send all payloads for one entity via the azure-eventhub SDK (AMQP).

    Batches respect the Event Hubs size limit; a payload too large for an empty batch
    raises rather than being silently dropped.

    Args:
        job: Ingest job dict.
        payloads: Transformed records to send.

    Returns:
        Number of events sent.
    """
    producer = EventHubProducerClient.from_connection_string(eventstream_conn)
    sent = 0
    with producer:
        batch = producer.create_batch()
        for payload in payloads:
            event = build_event(job, payload)
            try:
                batch.add(event)
            except ValueError:
                if len(batch) == 0:
                    raise RuntimeError(
                        f"Single event for {job['entity']} exceeds the batch size limit"
                    )
                producer.send_batch(batch)
                sent += len(batch)
                batch = producer.create_batch()
                batch.add(event)
        if len(batch) > 0:
            producer.send_batch(batch)
            sent += len(batch)
    return sent


def send_records_https(job: Dict[str, Any], payloads: List[Dict[str, str]]) -> int:
    """Send all payloads for one entity via the Event Hubs REST batch API (HTTPS :443).

    Each request is a JSON array of ``{"Body": <payload JSON>, "UserProperties": {...}}``
    posted with Content-Type application/vnd.microsoft.servicebus.json; UserProperties are
    ignored if sent as headers on batch requests, so they always ride in the body. Requests
    are chunked under HTTPS_BATCH_LIMIT_BYTES; a single oversized event raises rather than
    being silently dropped.

    Args:
        job: Ingest job dict.
        payloads: Transformed records to send.

    Returns:
        Number of events sent.

    Raises:
        RuntimeError: On a non-201 response or an event exceeding the size limit alone.
    """
    entity_url = (
        f"https://{EVENTHUB_NAMESPACE}.servicebus.windows.net/{EVENTHUB_ENTITY_PATH}"
    )
    post_url = f"{entity_url}/messages?timeout=60&api-version=2014-01"
    headers = {
        "Authorization": build_sas_token(entity_url),
        "Content-Type": "application/vnd.microsoft.servicebus.json",
    }

    def post_batch(session: requests.Session, rows: List[str]) -> None:
        response = session.post(
            post_url,
            data=("[" + ",".join(rows) + "]").encode("utf-8"),
            headers=headers,
            timeout=60,
        )
        if response.status_code != 201:
            raise RuntimeError(
                f"HTTPS batch send failed for {job['entity']}: HTTP {response.status_code} "
                f"- {response.text[:300]}"
            )

    sent = 0
    batch: List[str] = []
    batch_bytes = 2  # the enclosing "[" + "]"
    # One Session per entity: keep-alive reuses a single TCP+TLS connection across every
    # batch POST instead of paying a fresh handshake per ~900 KB request.
    with requests.Session() as http_session:
        for payload in payloads:
            message = json.dumps(
                {
                    "Body": json.dumps(payload, ensure_ascii=False),
                    "UserProperties": build_event_properties(job),
                },
                ensure_ascii=False,
            )
            message_bytes = len(message.encode("utf-8")) + 1  # +1 joining comma
            if message_bytes > HTTPS_BATCH_LIMIT_BYTES:
                raise RuntimeError(
                    f"Single event for {job['entity']} exceeds the batch size limit"
                )
            if batch and batch_bytes + message_bytes > HTTPS_BATCH_LIMIT_BYTES:
                post_batch(http_session, batch)
                sent += len(batch)
                batch, batch_bytes = [], 2
            batch.append(message)
            batch_bytes += message_bytes
        if batch:
            post_batch(http_session, batch)
            sent += len(batch)
    return sent


def send_records(job: Dict[str, Any], payloads: List[Dict[str, str]]) -> int:
    """Send all payloads for one entity over the transport selected by SEND_PROTOCOL.

    Args:
        job: Ingest job dict.
        payloads: Transformed records to send.

    Returns:
        Number of events sent.
    """
    if SEND_PROTOCOL == "https":
        return send_records_https(job, payloads)
    return send_records_amqp(job, payloads)

## Run (soft fail per entity)

One entity failing must not block the rest; failures keep their old watermark and retry
the same window next run. Watermarks are captured at run start and persisted only on
success, so overlap duplicates are possible by design - deduplicate downstream (e.g. KQL
`arg_max` materialized views or Delta merge). `DRY_RUN` reports what *would* be sent.

In [ ]:
known_entities = {j["entity"] for j in job_definitions}
unknown_filters = [e for e in ENTITY_FILTER if e not in known_entities]
if unknown_filters:
    raise RuntimeError(f"ENTITY_FILTER names match no ingest control record: {unknown_filters}")

sample_printed = False

for job in job_definitions:
    entity = job["entity"]
    if ENTITY_FILTER and entity not in ENTITY_FILTER:
        results["skipped"].append({"name": entity, "reason": "not in ENTITY_FILTER"})
        continue
    try:
        started = time.monotonic()
        since = job["last_watermark"] or datetime.fromisoformat(DEFAULT_WATERMARK)
        run_started = datetime.now(timezone.utc).replace(tzinfo=None)
        raw_records = load_ido_records(job, since)
        payloads = [transform_record(r, job) for r in raw_records]

        if DRY_RUN:
            print(f"[DRY_RUN] {entity}: {len(payloads):,} record(s) since "
                  f"{since:%Y-%m-%d %H:%M} ({time.monotonic() - started:.0f}s)")
            if payloads and not sample_printed:
                print("  sample envelope properties:")
                for key, value in build_event_properties(job).items():
                    print(f"    {key} = {value}")
                sample_printed = True
        else:
            sent = send_records(job, payloads)
            set_watermark(entity, run_started)
            elapsed = int(time.monotonic() - started)
            log_run(
                entity,
                "SUCCESS" if sent > 0 else "NOROWS",
                records_fetched=len(raw_records),
                records_sent=sent,
                duration_seconds=elapsed,
                watermark=run_started,
            )
            print(f"{entity}: sent {sent:,} event(s) in {elapsed}s; "
                  f"watermark -> {run_started:%Y-%m-%d %H:%M:%S}")
        results["succeeded"].append(entity)
    except Exception as exc:
        results["failed"].append({"name": entity, "error": str(exc)})
        if not DRY_RUN:
            log_run(entity, "FAILED",
                    duration_seconds=int(time.monotonic() - started),
                    failure_message=str(exc))

In [ ]:
# ── Summary ──────────────────────────────────────────────────────────────────────
print("── Summary ───────────────────────────────────")
print(f"  Mode:      {'DRY_RUN' if DRY_RUN else 'LIVE'}")
print(f"  Succeeded: {len(results['succeeded'])}")
print(f"  Skipped:   {len(results['skipped'])}")
print(f"  Failed:    {len(results['failed'])}")
for item in results["failed"]:
    print(f"    - {item['name']}: {item['error']}")

if STRICT and results["failed"]:
    raise RuntimeError(f"{len(results['failed'])} entity load(s) failed (STRICT mode)")

# Exit value for a pipeline: @activity('<NotebookActivity>').output.result.exitValue
exit_value = json.dumps({
    "mode": "DRY_RUN" if DRY_RUN else "LIVE",
    "runId": RUN_ID,
    "succeeded": len(results["succeeded"]),
    "skipped": len(results["skipped"]),
    "failed": len(results["failed"]),
    "failedEntities": [f["name"] for f in results["failed"]],
})
# exit() halts execution by raising an internal NotebookExit exception, so an interactive
# run renders this cell as errored even when everything succeeded. Only pipeline runs need
# the exit (it is what surfaces exitValue); interactive runs print the same JSON instead.
# Must stay outside any try/except or the exit is swallowed.
try:
    _is_pipeline_run = bool(notebookutils.runtime.context.get("isForPipeline"))
except Exception:
    _is_pipeline_run = bool(PIPELINE_RUN_ID)
if _is_pipeline_run and hasattr(notebookutils.notebook, "exit"):
    notebookutils.notebook.exit(exit_value)
else:
    print(exit_value)